# Tanzanian water pumps: corrected data audit

This notebook reviews the supplied data and documentation. It does not train or evaluate a prediction model.

Put `WaterPumps.csv` and `DataLabels.csv` beside this notebook. Set `DATA_DIR` below if they are elsewhere.
The ZIP package contains both CSV files and the original documentation.

The source has 59,400 rows and 40 columns: one identifier and 39 potential predictors.
The validated label merge adds one target column, giving 41 columns.
Descriptions use `Data Description.pdf`, pages 2-3. Target meanings use pages 5-6.
The PDF gives no definition for `num_private`. It gives the same operator description for both scheme fields.

Parsed nulls use pandas defaults. They include empty fields and literal `None` strings.
Explicit `unknown`, lowercase `none`, and zero values remain separate from parsed nulls.
The audit preserves the input values. A suspicious value is not automatically an error.


In [1]:
from pathlib import Path
import platform
import pandas as pd

DATA_DIR = Path('.')
df = pd.read_csv(DATA_DIR / 'WaterPumps.csv')
labels_df = pd.read_csv(DATA_DIR / 'DataLabels.csv')
raw_df = pd.read_csv(DATA_DIR / 'WaterPumps.csv', dtype=str, keep_default_na=False)

assert df.shape == (59400, 40), 'Unexpected source shape'
assert labels_df.shape == (59400, 2), 'Unexpected label shape'
assert set(labels_df.columns) == {'id', 'status_group'}
for frame in [df, labels_df]:
    assert frame['id'].notna().all(), 'Missing ID'
    assert frame['id'].is_unique, 'Duplicate ID'
assert set(df['id']) == set(labels_df['id']), 'Unmatched IDs'
full_df = df.merge(labels_df, on='id', how='left', validate='one_to_one')
assert len(full_df) == len(df)
assert full_df['status_group'].notna().all()
assert set(full_df['status_group']) == {'functional', 'non functional', 'functional needs repair'}
print(f'Python {platform.python_version()}; pandas {pd.__version__}')
print('Source:', df.shape, 'Labels:', labels_df.shape, 'Merged:', full_df.shape)
print('ID checks pass: unique, non-null, and identical sets.')


Python 3.14.2; pandas 3.0.5
Source: (59400, 40) Labels: (59400, 2) Merged: (59400, 41)
ID checks pass: unique, non-null, and identical sets.


In [2]:
counts = full_df['status_group'].value_counts(dropna=False)
target_summary = pd.DataFrame({'Count': counts, 'Percent': (counts / len(full_df) * 100).round(2)})
print(target_summary.to_string())


                         Count  Percent
status_group                           
functional               32259    54.31
non functional           22824    38.42
functional needs repair   4317     7.27


The class counts are 32,259 functional (54.31%), 22,824 non functional (38.42%), and 4,317 functional needs repair (7.27%).
The original counts and percentages are correct. `functional needs repair` means operational, but requiring repairs.

The following table covers all 41 columns. Storage types describe pandas 2.2.3 with default CSV parsing.
Semantic types describe the variables. An integer storage type does not make a geographic code a numeric measurement.


In [3]:
info_df = pd.DataFrame({
    'Data Type': full_df.dtypes.astype(str),
    'Missing Values': full_df.isna().sum(),
    'Percentage Missing (%)': full_df.isna().mean().mul(100).round(4),
    'Unique Values (non-null)': full_df.nunique(dropna=True)
})
print(info_df.to_string())

missing_detail = pd.DataFrame({
    'Parsed nulls': df.isna().sum(),
    'Raw empty fields': raw_df.eq('').sum(),
    'Raw literal None': raw_df.eq('None').sum()
})
print('\nMissing-value parsing details:')
print(missing_detail.loc[missing_detail['Parsed nulls'] > 0].to_string())


                      Data Type  Missing Values  Percentage Missing (%)  Unique Values (non-null)
id                        int64               0                  0.0000                     59400
amount_tsh              float64               0                  0.0000                        98
date_recorded               str               0                  0.0000                       356
funder                      str            3637                  6.1229                      1896
gps_height                int64               0                  0.0000                      2428
installer                   str            3655                  6.1532                      2145
longitude               float64               0                  0.0000                     57516
latitude                float64               0                  0.0000                     57517
wpt_name                    str               2                  0.0034                     37399
num_private         

| Attribute | pandas dtype | Semantic type | Description | Null count (%) | Distinct non-null | Quality notes | PDF page |
|---|---|---|---|---:|---:|---|---|
| `id` | int64 | Nominal identifier (integer storage) | Unique identifier for each waterpoint. | 0 (0.0000%) | 59,400 | 59,400 unique IDs. ID zero is valid as a key. Exclude the identifier from ordinary predictors. | CSV key; PDF p. 6 |
| `amount_tsh` | float64 | Continuous measurement | Total static head (amount water available to waterpoint). | 0 (0.0000%) | 98 | 41,639 zeros (70.10%). The PDF does not define zero as missing or specify units. Keep float64. | 2 |
| `date_recorded` | object | Temporal date (day resolution) | The date the row was entered. | 0 (0.0000%) | 356 | All dates parse with %m/%d/%Y. Range: 2002-10-14 to 2013-12-03. The PDF example uses ISO format. | 2 |
| `funder` | object | nominal | Who funded the well. | 3,637 (6.1229%) | 1,896 | 3,637 parsed nulls = 3,635 blanks + 2 literal None values. Also 777 literal 0 values and 4 Unknown values. | 2 |
| `gps_height` | int64 | Continuous measurement | Altitude of the well. | 0 (0.0000%) | 2,428 | 20,438 zeros; 1,496 negative values; range -90 to 2,770. These values alone do not prove missing data. | 2 |
| `installer` | object | nominal | Organization that installed the well. | 3,655 (6.1532%) | 2,145 | 3,655 blanks. Also 777 literal 0 values, 4 case-insensitive unknown values, and 3 hyphens. DWE and Dwe both occur. | 2 |
| `longitude` | float64 | Continuous measurement | GPS coordinate. | 0 (0.0000%) | 57,516 | 1,812 zeros. These rows also have latitude -0.00000002. Flag the coordinate pair for review. | 2 |
| `latitude` | float64 | Continuous measurement | GPS coordinate. | 0 (0.0000%) | 57,517 | No exact zeros. 1,812 values equal -0.00000002, paired with longitude zero. Suspected placeholders, not documented missing codes. | 2 |
| `wpt_name` | object | nominal | Name of the waterpoint if there is one. | 2 (0.0034%) | 37,399 | 2 literal None values become null. Lowercase none remains a category in 3,563 rows; Unknown occurs 3 times. | 2 |
| `num_private` | int64 | Integer-valued; meaning undocumented | No definition is supplied in the PDF. | 0 (0.0000%) | 65 | 65 distinct integer values; 58,643 zeros. Do not invent a definition from the column name. | 2 |
| `basin` | object | nominal | Geographic water basin. | 0 (0.0000%) | 9 | 9 categories. No parsed nulls. This does not prove that all values are correct. | 2 |
| `subvillage` | object | nominal | Geographic location. | 371 (0.6246%) | 19,287 | 371 blanks; 19,287 distinct non-null values. High cardinality alone does not prove spelling errors. | 2 |
| `region` | object | nominal | Geographic location. | 0 (0.0000%) | 21 | 21 categories. Region codes do not map consistently to names in this file. | 2 |
| `region_code` | int64 | coded nominal | Geographic location (coded). | 0 (0.0000%) | 27 | 27 codes and 21 region names. Codes 5, 11, 14, 17, and 18 each map to two region names. | 2 |
| `district_code` | int64 | coded nominal | Geographic location (coded). | 0 (0.0000%) | 20 | 20 codes; 23 zeros. Codes repeat across geographic areas. Do not treat them as numerical magnitudes. | 2 |
| `lga` | object | nominal | Geographic location. | 0 (0.0000%) | 125 | 125 distinct non-null values. | 2 |
| `ward` | object | nominal | Geographic location. | 0 (0.0000%) | 2,092 | 2,092 distinct non-null values. | 2 |
| `population` | int64 | Discrete count | Population around the well. | 0 (0.0000%) | 1,049 | 21,381 zeros (36.00%). The PDF does not define zero as missing. | 2 |
| `public_meeting` | object | Binary nominal (Boolean with missing values) | True/False. | 3,334 (5.6128%) | 2 | 3,334 blanks. Default pandas dtype is object because Boolean values coexist with nulls. | 2 |
| `recorded_by` | object | nominal | Group entering this row of data. | 0 (0.0000%) | 1 | Constant: GeoData Consultants Ltd. No variation for prediction in this file. | 2 |
| `scheme_management` | object | nominal | Who operates the waterpoint. | 3,878 (6.5286%) | 11 | 3,878 parsed nulls = 3,877 blanks + 1 literal None value. | 2 |
| `scheme_name` | object | nominal | Who operates the waterpoint. | 28,810 (48.5017%) | 2,695 | 28,810 parsed nulls (48.50%) = 28,166 blanks + 644 literal None values. Also 25 lowercase none values. | 2 |
| `permit` | object | Binary nominal (Boolean with missing values) | If the waterpoint is permitted. | 3,056 (5.1448%) | 2 | 3,056 blanks. Default pandas dtype is object because Boolean values coexist with nulls. | 3 |
| `construction_year` | int64 | Discrete temporal year | Year the waterpoint was constructed. | 0 (0.0000%) | 55 | 20,709 zeros (34.86%). Nonzero range: 1960-2013. Nine records postdate their recording year; investigate. | 3 |
| `extraction_type` | object | nominal | The kind of extraction the waterpoint uses. | 0 (0.0000%) | 18 | 18 categories. Each maps to one extraction_type_group in this file. | 3 |
| `extraction_type_group` | object | nominal | The kind of extraction the waterpoint uses. | 0 (0.0000%) | 13 | 13 categories. Each maps to one extraction_type_class in this file. | 3 |
| `extraction_type_class` | object | nominal | The kind of extraction the waterpoint uses. | 0 (0.0000%) | 7 | 7 categories. Coarser encoding of extraction_type_group in this file. | 3 |
| `management` | object | nominal | How the waterpoint is managed. | 0 (0.0000%) | 12 | 561 explicit unknown values (0.94%); zero parsed nulls. Each category maps to one management_group. | 3 |
| `management_group` | object | nominal | How the waterpoint is managed. | 0 (0.0000%) | 5 | 561 explicit unknown values; zero parsed nulls. Coarser encoding of management. | 3 |
| `payment` | object | nominal | What the water costs. | 0 (0.0000%) | 7 | 8,157 explicit unknown values (13.73%); zero parsed nulls. One-to-one category mapping with payment_type. | 3 |
| `payment_type` | object | nominal | What the water costs. | 0 (0.0000%) | 7 | 8,157 explicit unknown values (13.73%); zero parsed nulls. Categories describe payment arrangements, not numeric prices. | 3 |
| `water_quality` | object | nominal | The quality of the water. | 0 (0.0000%) | 8 | 1,876 explicit unknown values (3.16%); zero parsed nulls. Each maps to one quality_group. | 3 |
| `quality_group` | object | nominal | The quality of the water. | 0 (0.0000%) | 6 | 1,876 explicit unknown values; zero parsed nulls. Coarser encoding of water_quality. | 3 |
| `quantity` | object | Nominal as supplied; unknown has no order | The quantity of water. | 0 (0.0000%) | 5 | 789 explicit unknown values (1.33%); zero parsed nulls. Identical to quantity_group in every row. | 3 |
| `quantity_group` | object | Nominal as supplied; unknown has no order | The quantity of water. | 0 (0.0000%) | 5 | Exact duplicate of quantity in this file. | 3 |
| `source` | object | nominal | The source of the water. | 0 (0.0000%) | 10 | 66 explicit unknown values (0.11%); zero parsed nulls. Each category maps to one source_type. | 3 |
| `source_type` | object | nominal | The source of the water. | 0 (0.0000%) | 7 | 7 categories. Each category maps to one source_class. | 3 |
| `source_class` | object | nominal | The source of the water. | 0 (0.0000%) | 3 | 278 explicit unknown values (0.47%); zero parsed nulls. | 3 |
| `waterpoint_type` | object | nominal | The kind of waterpoint. | 0 (0.0000%) | 7 | 7 categories. Each maps to one waterpoint_type_group. No specific error is established by this audit. | 3 |
| `waterpoint_type_group` | object | nominal | The kind of waterpoint. | 0 (0.0000%) | 6 | 6 categories. Coarser encoding of waterpoint_type in this file. | 3 |
| `status_group` | object | target nominal | Operational without repairs; operational but needs repairs; or not operational. | 0 (0.0000%) | 3 | 32,259 functional; 22,824 non functional; 4,317 functional needs repair. No nulls. The last class remains operational. | 5-6 |

## Dates and suspicious numeric values

All 59,400 dates use month/day/year. Parse them explicitly with `%m/%d/%Y`.
The file differs from the ISO date example in the PDF.
Zero construction years cannot provide valid construction dates. Their intended missing-value meaning remains undocumented.
Zero amounts, zero populations, and non-positive heights require review. Do not replace all of them without a justified rule.


In [4]:
recorded_dates = pd.to_datetime(df['date_recorded'], format='%m/%d/%Y', errors='raise')
print('Date range:', recorded_dates.min().date(), 'to', recorded_dates.max().date())
numeric_checks = pd.DataFrame({
    'zero_count': df[['amount_tsh', 'gps_height', 'longitude', 'latitude', 'population', 'construction_year']].eq(0).sum()
})
numeric_checks['zero_percent'] = numeric_checks['zero_count'].div(len(df)).mul(100).round(2)
print(numeric_checks.to_string())
print('Negative gps_height:', int(df['gps_height'].lt(0).sum()))
paired_coordinates = df['longitude'].eq(0) & df['latitude'].eq(-2e-8)
print('Suspicious coordinate pairs:', int(paired_coordinates.sum()))
date_conflicts = df.loc[df['construction_year'] > recorded_dates.dt.year,
                        ['id', 'date_recorded', 'construction_year']]
print('Construction after recording year:', len(date_conflicts))
print(date_conflicts.to_string(index=False))


Date range: 2002-10-14 to 2013-12-03
                   zero_count  zero_percent
amount_tsh              41639         70.10
gps_height              20438         34.41
longitude                1812          3.05
latitude                    0          0.00
population              21381         35.99
construction_year       20709         34.86
Negative gps_height: 1496
Suspicious coordinate pairs: 1812
Construction after recording year: 9
   id date_recorded  construction_year
20198      5/1/2004               2008
55069     12/1/2004               2006
48759      6/1/2004               2011
20534      3/1/2004               2009
53086      1/7/2004               2009
 9195      3/1/2004               2005
 5971      3/1/2004               2007
15921      3/6/2004               2009
 1251     12/1/2004               2008


## Unknown categories and repeated information

Zero parsed nulls do not establish complete information. Several columns contain an explicit `unknown` category.
The mapping checks below apply to these supplied rows. They do not prove that the mappings hold for future data.
`quantity_group` duplicates `quantity` exactly. Group columns often provide a coarser version of another column.
Spelling normalization requires a reviewed mapping. Similar names alone do not establish the same organization.


In [5]:
print('Explicit unknown categories, ignoring case:')
for col in raw_df:
    count = int(raw_df[col].str.casefold().eq('unknown').sum())
    if count:
        print(f'{col}: {count:,} ({count / len(df) * 100:.2f}%)')

pairs = [
    ('extraction_type', 'extraction_type_group'),
    ('extraction_type_group', 'extraction_type_class'),
    ('management', 'management_group'), ('payment', 'payment_type'),
    ('water_quality', 'quality_group'), ('quantity', 'quantity_group'),
    ('source', 'source_type'), ('source_type', 'source_class'),
    ('waterpoint_type', 'waterpoint_type_group')
]
print('Maximum target categories per source category:')
for source_col, target_col in pairs:
    maximum = int(df.groupby(source_col)[target_col].nunique(dropna=False).max())
    print(f'{source_col} -> {target_col}: {maximum}')
print('quantity equals quantity_group:', df['quantity'].equals(df['quantity_group']))
print('Constant columns:', [c for c in df if df[c].nunique(dropna=False) == 1])
region_names_per_code = df.groupby('region_code')['region'].unique()
print('Region codes with multiple names:')
print(region_names_per_code[region_names_per_code.map(len) > 1].to_string())


Explicit unknown categories, ignoring case:
funder: 4 (0.01%)
installer: 4 (0.01%)
wpt_name: 3 (0.01%)
management: 561 (0.94%)
management_group: 561 (0.94%)
payment: 8,157 (13.73%)
payment_type: 8,157 (13.73%)
water_quality: 1,876 (3.16%)
quality_group: 1,876 (3.16%)
quantity: 789 (1.33%)
quantity_group: 789 (1.33%)
source: 66 (0.11%)
source_class: 278 (0.47%)
Maximum target categories per source category:
extraction_type -> extraction_type_group: 1
extraction_type_group -> extraction_type_class: 1
management -> management_group: 1
payment -> payment_type: 1
water_quality -> quality_group: 1
quantity -> quantity_group: 1
source -> source_type: 1
source_type -> source_class: 1
waterpoint_type -> waterpoint_type_group: 1
quantity equals quantity_group: True
Constant columns: ['recorded_by']
Region codes with multiple names:
region_code
5       [Morogoro, Tanga]
11    [Iringa, Shinyanga]
14    [Tabora, Shinyanga]
17    [Shinyanga, Mwanza]
18        [Kagera, Lindi]


## Scope and correction notes

The original notebook executes after its Colab paths change to the supplied file locations.
Its saved output matches the reproduced output exactly under pandas 2.2.3.
The guarded `value_value_counts` typo does not cause a failure. The original code falls back to `value_counts`.
This version uses `value_counts` directly, validates the merge, and prints every column.

The original table omits 21 source columns. This table includes them.
Missing counts use an explicit parsing convention. Quality notes distinguish observed values from proposed explanations.
No model, train/test split, metric, or prediction appears in the original notebook.
Therefore, this review establishes data-summary accuracy and execution, not predictive accuracy.
